In [ ]:
import os; os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
importlib.reload(parametric_pillows)
# m, fuseMarkers = inflation.triangulate_channel_walls(*parametric_pillows.squareWithVerticalChannels(0.9, 8), 0.001)
m, fuseMarkers, fuseEdges = wall_generation.triangulate_channel_walls(*parametric_pillows.concentricCircles(8, 50), 0.001)
# visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=12, height=12)

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)
isheet.setIdentityDeformation(prepareRigidMotionPins=True)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
opts.factorizer = opts.factorizer.CatamariNesdis

In [ ]:
import time, vis
benchmark.reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
opts.niter = 5000
# isheet.pressure = 5.75 * 3.75
isheet.pressure = 20 * 3.75

# Choose strategy for constraining rigid motion
fixedVars, hessianShift = isheet.rigidMotionPinVars, 0
fixedVars, hessianShift = [], 1e-6

def cb(it):
    if it % 20 == 0:
        viewer.update()
    return False
cr = inflation.inflation_newton(isheet, fixedVars, opts, cb, hessianShift = hessianShift)
benchmark.report()

### Repeat the inflation, this time recording it to a video

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

from tri_mesh_viewer import OffscreenTriMeshViewer
oview = OffscreenTriMeshViewer(isheet, width=768, height=640, wireframe=True)

benchmark.reset()
opts.niter=1000
oview.recordStart('cc_inflate.mp4')
isheet.pressure = 20
cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts,
                                callback=lambda it: oview.update())
benchmark.report()
oview.recordStop()

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
strains = utils.getStrains(isheet)[:, 0]
plt.hist(strains, 60);
plt.xlabel('Principal stretch $\\lambda_0$')
print(np.median(strains))

In [ ]:
isa = inflation.InflatedSurfaceAnalysis(isheet)
curvature = isa.curvature()
metric = isa.metric()

In [ ]:
import matplotlib, vis
from tri_mesh_viewer import TriMeshViewer
metric_vf = vis.fields.VectorField(metric.sigma_2[:, None] * metric.left_stretch, vmin=0, vmax=1.0,
                                   align=vis.fields.VectorAlignment.CENTER, colormap=matplotlib.cm.viridis,
                                   glyph=vis.fields.VectorGlyph.CYLINDER)

viewer2 = TriMeshViewer(isa.inflatedSurface(), width=768, height=640, scalarField=vis.fields.ScalarField(curvature.meanCurvature(), colormap=matplotlib.cm.coolwarm), vectorField=metric_vf)
viewer2.showWireframe()
viewer2.show()

In [ ]:
inflation.benchmark_reset()
isheet.setUseTensionFieldEnergy(False)
niter = 5000
iterations_per_output = 2
opts.niter = iterations_per_output
isheet.pressure = 30
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)
    if cr.numIters() < iterations_per_output: break
    isheet.writeDebugMesh('concentric_circles/inflation_non_tf_step_{}.msh'.format(step))
inflation.benchmark_report()

In [ ]:
norm(isheet.gradient())

In [ ]:
isheet.tensionStateHistogram()

In [ ]:
isheet.setUseHessianProjectedEnergy(True)

In [ ]:
fd_validation.validateHessian(isheet, xeval=isheet.getVars(), fd_eps=1e-7,
                              etype=isheet.EnergyType.Full)

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)
opts.niter = 1
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
niter = 5000
iterations_per_output = 10
opts.niter = iterations_per_output
isheet.pressure = 30
inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)

In [ ]:
isheet.setVars(x)
viewer2 = TriMeshViewer(isheet.visualizationMesh(), width=768, height=640)
viewer2.show()

In [ ]:
L, V = eigsh(H, 15, sigma=-1, which='LM')

In [ ]:
L

In [ ]:
x = isheet.getVars()

In [ ]:
isheet.setVars(x)

In [ ]:
isheet.setVars(x + 5 * V[:,10])
viewer2.update(preserveExisting=True, mesh=isheet.visualizationMesh())